# Ingestion Fabric en mode pull depuis API Inference

Ce notebook lit les événements en attente via l'API inference, écrit dans les tables Lakehouse, puis acquitte les événements traités.

In [ ]:
import requests
from urllib.parse import urlparse
from pyspark.sql import functions as F

# ─────────────────────────────────────────────────────────────
# Comment obtenir l'URL ngrok ?
#
# Option A — Docker (recommandé) :
#   Depuis Modeles_TimesSeries/ :
#     docker compose --profile local up -d
#   Puis récupère l'URL :
#     docker logs ngrok-tunnel 2>&1 | findstr "url="
#   Ou ouvre : http://127.0.0.1:4040 dans ton navigateur
#
# Option B — ngrok manuel :
#     C:\ngrok\ngrok.exe http 8001
#   L'URL s'affiche dans le terminal.
# ─────────────────────────────────────────────────────────────

API_BASE_URL = "https://REMPLACE-PAR-TON-URL-NGROK"
API_KEY = "dev-inference-key"   # valeur INFERENCE_API_KEY dans api-inference/.env
PENDING_LIMIT = 500

# ── Validation ───────────────────────────────────────────────
if "REMPLACE" in API_BASE_URL or "<" in API_BASE_URL or ">" in API_BASE_URL:
    raise ValueError("Remplacez API_BASE_URL par ton URL ngrok réelle (ex: https://abcd-1234.ngrok-free.app)")

parsed = urlparse(API_BASE_URL)
if parsed.scheme not in {"http", "https"} or not parsed.netloc:
    raise ValueError("API_BASE_URL invalide. Exemple: https://xxxx.ngrok-free.app")

HEADERS = {"X-API-Key": API_KEY}
print(f"✅ Config chargée: {API_BASE_URL}")

In [ ]:
pending_url = f"{API_BASE_URL}/fabric-exports/pending"
resp = requests.get(pending_url, params={"limit": PENDING_LIMIT}, headers=HEADERS, timeout=60)
resp.raise_for_status()
payload = resp.json()
items = payload.get("items", [])
print(f"Total pending côté API: {payload.get('total_pending', 0)}")
print(f"Événements récupérés: {len(items)}")

In [ ]:
if not items:
    print("Aucun événement à traiter")
else:
    df = spark.createDataFrame(items)
    display(df.limit(20))

In [ ]:
if items:
    models_df = (
        spark.createDataFrame(items)
        .select(
            F.col("model.model_id").alias("ID_MODELE"),
            F.col("prm").alias("PRM"),
            F.col("model.model_name").alias("NOM_MODELE"),
            F.col("model.model_version").alias("VERSION_MODELE"),
            F.col("model.artifact_uri").alias("URI"),
            F.to_timestamp(F.col("model.created_at")).alias("DATE_CREATION"),
            F.when(F.col("model.is_active") == True, F.lit(1)).otherwise(F.lit(0)).alias("IS_ACTIVE")
        )
        .dropDuplicates(["ID_MODELE"])
    )

    models_df.write.mode("append").saveAsTable("ia_modeles")
    print(f"ia_modeles: {models_df.count()} ligne(s) ajoutée(s)")

In [ ]:
if items:
    metrics_source_df = spark.createDataFrame(items).select(
        F.col("model.model_id").alias("ID_MODELE"),
        F.col("prm").alias("PRM"),
        F.to_timestamp(F.col("measured_at")).alias("DATE_MESURE"),
        F.col("metrics").alias("METRICS_MAP")
    )

    metrics_df = (
        metrics_source_df
        .select(
            F.expr("uuid()").alias("ID_METRIC"),
            F.col("ID_MODELE"),
            F.col("PRM"),
            F.explode_outer(F.col("METRICS_MAP")).alias("TYPE_METRIC", "VALEUR_METRIC"),
            F.col("DATE_MESURE")
        )
        .where(F.col("TYPE_METRIC").isNotNull())
        .withColumn("VALEUR_METRIC", F.col("VALEUR_METRIC").cast("double"))
    )

    metrics_df.write.mode("append").saveAsTable("ia_metrics")
    print(f"ia_metrics: {metrics_df.count()} ligne(s) ajoutée(s)")

In [ ]:
if items:
    event_ids = [it.get("event_id") for it in items if it.get("event_id")]
    ack_url = f"{API_BASE_URL}/fabric-exports/ack"
    ack_resp = requests.post(ack_url, headers=HEADERS, json={"event_ids": event_ids}, timeout=60)
    ack_resp.raise_for_status()
    print("ACK result:", ack_resp.json())

## Notes d'exploitation
- Planifier ce notebook (Fabric scheduler) toutes les 5 à 15 minutes.
- Vérifier que l'URL de l'API inference est accessible depuis Fabric (VPN/tunnel/public endpoint).
- Conserver l'API key dans un secret/capacité Fabric plutôt qu'en clair.
- Depuis Fabric cloud, `localhost`/`127.0.0.1` pointent vers Fabric, pas vers votre PC.
- Exemple d'URL valide: `https://xxxx.ngrok-free.app` (sans `/` final).